# Notebook 08 — Recommendations

Synthesise findings from all analyses into a prioritised, actionable refactoring plan.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from build_optimiser.graph import (
    load_graph, attach_metrics, critical_path, critical_path_length, node_centrality
)
from build_optimiser.simulation import (
    rebuild_cost, expected_daily_cost, simulate_merge, simulate_split,
    schema_change_probability, schema_pain_score,
)
from build_optimiser.config import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")

file_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "file_metrics.parquet")
target_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "target_metrics.parquet")
edge_list = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "edge_list.parquet")

G = load_graph(str(PROJECT_ROOT / "data" / "raw" / "dot"))
attach_metrics(G, target_metrics)

# Load codegen inventory
codegen_path = PROJECT_ROOT / "data" / "raw" / "codegen_inventory.csv"
codegen_inv = pd.read_csv(codegen_path) if codegen_path.exists() else pd.DataFrame()

print(f"Files: {len(file_metrics)}, Targets: {len(target_metrics)}, Edges: {len(edge_list)}")
print(f"Codegen invocations: {len(codegen_inv)}")

## Compile Candidate List

Merge candidates from community detection, split candidates from centrality analysis,
and individual target optimisation opportunities.

In [ ]:
centrality = node_centrality(G)
cp = critical_path(G, weight_attr='compile_time_sum_ms')
cp_set = set(cp)
cp_len = critical_path_length(G, weight_attr='compile_time_sum_ms')

# Build a unified scoring DataFrame
candidates = target_metrics.copy()
candidates['betweenness_centrality'] = candidates['cmake_target'].map(centrality).fillna(0)
candidates['on_critical_path'] = candidates['cmake_target'].isin(cp_set)

# Pain score: expected rebuild cost
candidates['pain_score'] = candidates.apply(
    lambda row: rebuild_cost(G, row['cmake_target'], target_metrics)
    if row['cmake_target'] in G else 0,
    axis=1
)

print(f"Critical path length: {cp_len:.0f} ms")
print(f"Critical path targets: {len(cp)}")
print(f"\nTop 10 by pain score:")
candidates.nlargest(10, 'pain_score')[['cmake_target', 'compile_time_sum_ms',
    'pain_score', 'betweenness_centrality', 'on_critical_path']]

## Score Each Candidate

Compute expected build time improvement for each proposed action.
Factor in estimated implementation effort.

In [ ]:
recommendations = []

# --- Split candidates: high centrality targets with many files ---
split_cands = candidates[
    (candidates['betweenness_centrality'] > candidates['betweenness_centrality'].quantile(0.75)) &
    (candidates['file_count'] > 3)
].copy()

for _, row in split_cands.iterrows():
    target = row['cmake_target']
    # Estimated improvement: splitting reduces rebuild scope by ~30-50%
    estimated_improvement_ms = row['pain_score'] * 0.3
    # Effort heuristic: proportional to SLOC
    effort = row.get('code_lines_total', 100) * 0.1  # person-hours
    
    recommendations.append({
        'action': 'split',
        'target': target,
        'estimated_daily_savings_ms': estimated_improvement_ms,
        'estimated_effort_hours': max(effort, 1),
        'on_critical_path': row['on_critical_path'],
        'rationale': f"High centrality ({row['betweenness_centrality']:.3f}), "
                     f"{row['file_count']} files, pain={row['pain_score']:.0f}ms",
    })

# --- Critical path optimisation: targets on the critical path ---
for _, row in candidates[candidates['on_critical_path']].iterrows():
    target = row['cmake_target']
    if target not in [r['target'] for r in recommendations]:
        estimated_improvement_ms = row['compile_time_sum_ms'] * 0.2  # Header cleanup
        recommendations.append({
            'action': 'optimise_headers',
            'target': target,
            'estimated_daily_savings_ms': estimated_improvement_ms,
            'estimated_effort_hours': max(row.get('code_lines_total', 100) * 0.02, 1),
            'on_critical_path': True,
            'rationale': f"On critical path, compile_time={row['compile_time_sum_ms']:.0f}ms, "
                         f"header_depth={row.get('header_depth_max', 0)}",
        })

rec_df = pd.DataFrame(recommendations)
print(f"Generated {len(rec_df)} recommendations")
rec_df.head(10)

## Rank by ROI

Sort by (expected build time saved per day) / (estimated implementation effort).

In [ ]:
if not rec_df.empty:
    rec_df['roi'] = rec_df['estimated_daily_savings_ms'] / rec_df['estimated_effort_hours']
    rec_df = rec_df.sort_values('roi', ascending=False).reset_index(drop=True)
    rec_df.index += 1  # 1-based ranking
    rec_df.index.name = 'rank'
    
    print("Recommendations ranked by ROI (ms saved per day / effort hours):")
    print("=" * 80)
    rec_df[['action', 'target', 'estimated_daily_savings_ms', 'estimated_effort_hours',
            'roi', 'on_critical_path']].head(20)

## Quick Wins

Low-effort improvements that can be done immediately.

In [ ]:
quick_wins = []

# 1. Unused dependencies: edges in graph with no actual #include backing
# (Would need include analysis data to detect properly)
# Heuristic: targets with many dependencies but low preprocessed size
if 'preprocessed_bytes_total' in candidates.columns and 'direct_dependency_count' in candidates.columns:
    ratio = candidates['preprocessed_bytes_total'] / (candidates['direct_dependency_count'] + 1)
    low_use = candidates[ratio < ratio.quantile(0.1)]
    for _, row in low_use.iterrows():
        quick_wins.append({
            'type': 'remove_unused_deps',
            'target': row['cmake_target'],
            'detail': f"{row['direct_dependency_count']} deps, "
                      f"{row['preprocessed_bytes_total']} preprocessed bytes",
        })

# 2. High preprocessed size on critical path (include-what-you-use candidates)
cp_targets = candidates[candidates['on_critical_path']]
if 'preprocessed_bytes_total' in cp_targets.columns:
    high_preproc = cp_targets.nlargest(5, 'preprocessed_bytes_total')
    for _, row in high_preproc.iterrows():
        quick_wins.append({
            'type': 'include_what_you_use',
            'target': row['cmake_target'],
            'detail': f"Critical path, {row['preprocessed_bytes_total']} preprocessed bytes",
        })

# 3. High header depth (precompiled header candidates)
if 'header_depth_max' in candidates.columns:
    deep_headers = candidates.nlargest(5, 'header_depth_max')
    for _, row in deep_headers.iterrows():
        quick_wins.append({
            'type': 'precompiled_header',
            'target': row['cmake_target'],
            'detail': f"Max header depth: {row['header_depth_max']}",
        })

qw_df = pd.DataFrame(quick_wins)
print(f"Quick wins identified: {len(qw_df)}")
if not qw_df.empty:
    qw_df

## Generator-Level Optimisations

Recommendations targeting generated code specifically: protobuf lite runtime,
schema splitting, generator flag tuning, and custom generator output improvements.

In [ ]:
# ── Generator-level optimisations ────────────────────────────────────
codegen_recs = []

if not codegen_inv.empty and "is_generated" in file_metrics.columns:
    gen_files = file_metrics[file_metrics["is_generated"] == True]

    # 1. Protobuf: suggest lite runtime if protoc-generated code dominates
    if "protoc" in gen_files["generator"].values:
        proto_targets = gen_files[gen_files["generator"] == "protoc"].groupby("cmake_target").agg(
            gen_compile_ms=("compile_time_ms", "sum"),
        ).reset_index()
        proto_targets = proto_targets.merge(
            target_metrics[["cmake_target", "compile_time_sum_ms"]], on="cmake_target", how="left"
        )
        proto_targets["gen_fraction"] = proto_targets["gen_compile_ms"] / proto_targets["compile_time_sum_ms"]
        heavy_proto = proto_targets[proto_targets["gen_fraction"] > 0.3].sort_values("gen_compile_ms", ascending=False)
        for _, row in heavy_proto.iterrows():
            codegen_recs.append({
                "action": "protobuf_lite_runtime",
                "target": row["cmake_target"],
                "estimated_daily_savings_ms": row["gen_compile_ms"] * 0.4,
                "estimated_effort_hours": 4,
                "rationale": f"Protobuf generated code = {row['gen_fraction']:.0%} of compile time "
                             f"({row['gen_compile_ms']:.0f}ms). Switching to lite runtime can reduce by ~40%.",
            })

    # 2. Bison/flex: check compilation flags
    for gen_name in ["bison", "flex"]:
        if gen_name in gen_files["generator"].values:
            gen_subset = gen_files[gen_files["generator"] == gen_name]
            total_gen_ct = gen_subset["compile_time_ms"].sum()
            if total_gen_ct > 0:
                codegen_recs.append({
                    "action": f"{gen_name}_flag_review",
                    "target": "multiple",
                    "estimated_daily_savings_ms": total_gen_ct * 0.15,
                    "estimated_effort_hours": 2,
                    "rationale": f"{gen_name} generated code totals {total_gen_ct:.0f}ms compile time. "
                                 f"Review if generated parsers are compiled with unnecessary optimisation flags.",
                })

    # 3. Custom generators: flag where generated code dominates
    for gen_name in ["MessageCompiler", "DbAutoGen", "TemplateCompiler"]:
        if gen_name in gen_files["generator"].values:
            gen_subset = gen_files[gen_files["generator"] == gen_name]
            by_target = gen_subset.groupby("cmake_target")["compile_time_ms"].sum().reset_index()
            by_target = by_target.merge(
                target_metrics[["cmake_target", "compile_time_sum_ms"]], on="cmake_target", how="left"
            )
            by_target["gen_frac"] = by_target["compile_time_ms"] / by_target["compile_time_sum_ms"]
            heavy = by_target[by_target["gen_frac"] > 0.5].sort_values("compile_time_ms", ascending=False)
            for _, row in heavy.head(3).iterrows():
                codegen_recs.append({
                    "action": f"optimise_{gen_name}_output",
                    "target": row["cmake_target"],
                    "estimated_daily_savings_ms": row["compile_time_ms"] * 0.2,
                    "estimated_effort_hours": 8,
                    "rationale": f"{gen_name} output = {row['gen_frac']:.0%} of compile time. "
                                 f"Consider producing leaner generated output.",
                })

    # 4. Schema splitting recommendations (from pain scores)
    git_df = file_metrics[["source_file", "git_commit_count"]].drop_duplicates() \
        if "git_commit_count" in file_metrics.columns else pd.DataFrame()
    if not git_df.empty:
        schema_probs = schema_change_probability(git_df, codegen_inv)
        pain = schema_pain_score(schema_probs, file_metrics, codegen_inv)
        high_pain = pain[pain["pain_score"] > pain["pain_score"].quantile(0.8)] if not pain.empty else pd.DataFrame()
        for _, row in high_pain.head(5).iterrows():
            codegen_recs.append({
                "action": "split_schema",
                "target": Path(row["input_file"]).name,
                "estimated_daily_savings_ms": row["pain_score"] * 0.3,
                "estimated_effort_hours": 6,
                "rationale": f"Schema {Path(row['input_file']).name} ({row['generator']}): "
                             f"changed {row['commit_count']} times, affects {row['affected_target_count']} targets, "
                             f"pain_score={row['pain_score']:.0f}. Split into narrower schemas.",
            })

codegen_rec_df = pd.DataFrame(codegen_recs)

if not codegen_rec_df.empty:
    codegen_rec_df["roi"] = codegen_rec_df["estimated_daily_savings_ms"] / codegen_rec_df["estimated_effort_hours"]
    codegen_rec_df = codegen_rec_df.sort_values("roi", ascending=False).reset_index(drop=True)
    print(f"Generator-level recommendations: {len(codegen_rec_df)}")
    display(codegen_rec_df)

    # Merge into main recommendations
    if not rec_df.empty:
        codegen_rec_df["on_critical_path"] = False  # Will be overridden if applicable
        rec_df = pd.concat([rec_df, codegen_rec_df], ignore_index=True)
        rec_df["roi"] = rec_df["estimated_daily_savings_ms"] / rec_df["estimated_effort_hours"]
        rec_df = rec_df.sort_values("roi", ascending=False).reset_index(drop=True)
        rec_df.index += 1
        rec_df.index.name = "rank"
        print(f"\nTotal recommendations (including generator-level): {len(rec_df)}")
else:
    print("No generator-level recommendations (codegen inventory not available or no issues found).")

## Visualisation: Recommendation Impact

In [ ]:
if not rec_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Top 15 by ROI
    top15 = rec_df.head(15)
    colors = ['#e74c3c' if cp else '#3498db' for cp in top15['on_critical_path']]
    axes[0].barh(range(len(top15)), top15['roi'], color=colors)
    axes[0].set_yticks(range(len(top15)))
    axes[0].set_yticklabels([f"{r['action']}: {r['target']}" for _, r in top15.iterrows()], fontsize=8)
    axes[0].set_xlabel('ROI (ms saved / effort hour)')
    axes[0].set_title('Top 15 Recommendations by ROI')
    axes[0].invert_yaxis()
    
    # Savings vs effort scatter
    scatter_colors = ['#e74c3c' if cp else '#3498db' for cp in rec_df['on_critical_path']]
    axes[1].scatter(rec_df['estimated_effort_hours'], rec_df['estimated_daily_savings_ms'],
                    c=scatter_colors, alpha=0.6, s=50)
    axes[1].set_xlabel('Estimated Effort (hours)')
    axes[1].set_ylabel('Estimated Daily Savings (ms)')
    axes[1].set_title('Effort vs Impact')
    
    # Add labels for top candidates
    for _, row in rec_df.head(5).iterrows():
        axes[1].annotate(row['target'], (row['estimated_effort_hours'], row['estimated_daily_savings_ms']),
                        fontsize=7, ha='left')
    
    plt.tight_layout()
    plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'recommendation_charts.png'), dpi=150, bbox_inches='tight')
    plt.show()

## Output: Recommendations CSV and Report

In [ ]:
results_dir = PROJECT_ROOT / 'data' / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

# Save recommendations CSV
if not rec_df.empty:
    rec_df.to_csv(results_dir / 'recommendations.csv', index=True)
    print(f"Saved {len(rec_df)} recommendations to data/results/recommendations.csv")

# Save quick wins
if not qw_df.empty:
    qw_df.to_csv(results_dir / 'quick_wins.csv', index=False)
    print(f"Saved {len(qw_df)} quick wins to data/results/quick_wins.csv")

# Generate markdown report
report_lines = [
    '# Build Optimisation Recommendations',
    '',
    '## Summary',
    '',
    f'- **Total targets analysed:** {len(target_metrics)}',
    f'- **Total edges in dependency graph:** {len(edge_list)}',
    f'- **Critical path length:** {cp_len:.0f} ms',
    f'- **Targets on critical path:** {len(cp)}',
    f'- **Recommendations generated:** {len(rec_df)}',
    f'- **Quick wins identified:** {len(qw_df)}',
    '',
    '## Top Recommendations (by ROI)',
    '',
    '| Rank | Action | Target | Daily Savings (ms) | Effort (hrs) | ROI | Critical Path? |',
    '|------|--------|--------|--------------------|--------------|-----|----------------|',
]

if not rec_df.empty:
    for idx, row in rec_df.head(20).iterrows():
        report_lines.append(
            f"| {idx} | {row['action']} | {row['target']} | "
            f"{row['estimated_daily_savings_ms']:.0f} | "
            f"{row['estimated_effort_hours']:.1f} | "
            f"{row['roi']:.1f} | "
            f"{'Yes' if row['on_critical_path'] else 'No'} |"
        )

report_lines.extend([
    '',
    '## Quick Wins',
    '',
])

if not qw_df.empty:
    for _, row in qw_df.iterrows():
        report_lines.append(f"- **{row['type']}** — `{row['target']}`: {row['detail']}")

report = '\n'.join(report_lines)
report_path = results_dir / 'report.md'
report_path.write_text(report)
print(f"\nReport written to {report_path}")
print("\n" + report)